# CLIP: Learning Transferable Visual Models from Natural Language Supervision

## Learning Objectives
1. Understand contrastive vision-language learning and why it enables zero-shot transfer
2. Implement NT-Xent loss from scratch for multimodal contrastive learning
3. Build zero-shot classification and image-text retrieval using pre-trained CLIP
4. Analyze the effect of prompt engineering, batch size, and temperature on CLIP performance

## Cell 2: Imports and Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

# Try importing transformers; if not available, install
try:
    from transformers import CLIPProcessor, CLIPModel
except ImportError:
    print("Installing transformers...")
    import subprocess
    subprocess.check_call(["pip", "install", "transformers", "-q"])
    from transformers import CLIPProcessor, CLIPModel

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

## Level 1: Contrastive Vision-Language Loss (NumPy)

Implement the NT-Xent (Normalized Temperature-Scaled Cross-Entropy) loss that CLIP uses.
For a batch of N image-text pairs, create an N×N similarity matrix.
Positive: (image_i, text_i). Negatives: all other pairs.

In [ ]:
def cosine_similarity_matrix(X, Y):
    """Compute cosine similarity between X and Y (both L2-normalized).
    
    Args:
        X: (N, D) array
        Y: (M, D) array
    
    Returns:
        sim: (N, M) similarity matrix
    """
    # Assuming X and Y are already L2-normalized
    return X @ Y.T  # (N, M)


def clip_loss_numpy(image_embeddings, text_embeddings, tau=0.07):
    """CLIP contrastive loss (symmetric: image→text and text→image).
    
    Args:
        image_embeddings: (N, D) L2-normalized
        text_embeddings: (N, D) L2-normalized
        tau: temperature
    
    Returns:
        loss: float scalar
    """
    N = image_embeddings.shape[0]
    
    # Normalize embeddings
    img_norm = image_embeddings / (np.linalg.norm(image_embeddings, axis=1, keepdims=True) + 1e-8)
    txt_norm = text_embeddings / (np.linalg.norm(text_embeddings, axis=1, keepdims=True) + 1e-8)
    
    # Similarity matrix: (N, N)
    logits = cosine_similarity_matrix(img_norm, txt_norm) / tau
    
    # Labels: diagonal (i, i) are positive pairs
    labels = np.arange(N)
    
    # Image→Text loss
    losses_i2t = []
    for i in range(N):
        # Softmax over text dimension
        logits_row = logits[i]  # (N,)
        # Log-sum-exp for numerical stability
        max_logit = np.max(logits_row)
        exp_logits = np.exp(logits_row - max_logit)
        log_partition = max_logit + np.log(np.sum(exp_logits))
        loss_i = -(logits_row[i] - log_partition)
        losses_i2t.append(loss_i)
    
    # Text→Image loss (symmetric)
    losses_t2i = []
    logits_t = logits.T  # (N, N) but now text→image
    for i in range(N):
        logits_row = logits_t[i]
        max_logit = np.max(logits_row)
        exp_logits = np.exp(logits_row - max_logit)
        log_partition = max_logit + np.log(np.sum(exp_logits))
        loss_i = -(logits_row[i] - log_partition)
        losses_t2i.append(loss_i)
    
    # Average both directions
    return (np.mean(losses_i2t) + np.mean(losses_t2i)) / 2


# Demo: small batch of synthetic embeddings
N = 8
D = 16

# Randomly initialized embeddings
img_emb_random = np.random.randn(N, D).astype(np.float32)
txt_emb_random = np.random.randn(N, D).astype(np.float32)
loss_random = clip_loss_numpy(img_emb_random, txt_emb_random, tau=0.07)
print(f"[Random embeddings] CLIP loss: {loss_random:.4f}")

# Perfectly aligned embeddings (image_i ≈ text_i)
base_emb = np.random.randn(N, D).astype(np.float32)
img_emb_aligned = base_emb + np.random.randn(N, D) * 0.01  # Slight perturbation
txt_emb_aligned = base_emb + np.random.randn(N, D) * 0.01
loss_aligned = clip_loss_numpy(img_emb_aligned, txt_emb_aligned, tau=0.07)
print(f"[Aligned embeddings]   CLIP loss: {loss_aligned:.4f}")

# Measure loss vs alignment quality
print("\nLoss vs alignment noise level:")
for noise_std in [2.0, 1.0, 0.5, 0.1, 0.01]:
    base = np.random.randn(N, D).astype(np.float32)
    img_e = base + np.random.randn(N, D) * noise_std
    txt_e = base + np.random.randn(N, D) * noise_std
    loss = clip_loss_numpy(img_e, txt_e, tau=0.07)
    print(f"  Noise std {noise_std:.2f} -> loss {loss:.4f}")

## Level 2: CLIP-Style Training on Synthetic Image-Text Pairs (PyTorch)

Build a simple vision + text encoder pair, train them end-to-end with contrastive loss.
Visualize learned alignment in embedding space.

In [ ]:
# Synthetic data: 4 image classes (clusters), 4 text classes with aligned captions
class SyntheticImageTextDataset:
    def __init__(self, n_per_class=32, n_classes=4):
        # Generate synthetic "images" as 2D points
        centers = [[2.0, 2.0], [-2.0, 2.0], [-2.0, -2.0], [2.0, -2.0]]
        images, captions, labels = [], [], []
        for cls_idx in range(n_classes):
            cx, cy = centers[cls_idx % len(centers)]
            # Image cluster (2D points)
            pts = np.random.randn(n_per_class, 2) * 0.5 + [cx, cy]
            # Text embedding cluster (independent, will be aligned by training)
            text_pts = np.random.randn(n_per_class, 2) * 0.5
            for i in range(n_per_class):
                images.append(pts[i].astype(np.float32))
                captions.append(text_pts[i].astype(np.float32))
                labels.append(cls_idx)
        
        self.images = torch.tensor(np.array(images), dtype=torch.float32)
        self.captions = torch.tensor(np.array(captions), dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        return {
            'image': self.images[idx],
            'caption': self.captions[idx],
            'label': self.labels[idx]
        }


class VisionEncoder(nn.Module):
    """Simple vision encoder: 2D points → embedding."""
    def __init__(self, input_dim=2, hidden_dim=16, embedding_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim),
        )
    
    def forward(self, x):
        h = self.encoder(x)
        # L2 normalize
        return F.normalize(h, dim=-1)


class TextEncoder(nn.Module):
    """Simple text encoder: 2D caption vector → embedding."""
    def __init__(self, input_dim=2, hidden_dim=16, embedding_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim),
        )
    
    def forward(self, x):
        h = self.encoder(x)
        return F.normalize(h, dim=-1)


def clip_loss_torch(img_emb, txt_emb, tau=0.07):
    """CLIP loss in PyTorch (symmetric)."""
    N = img_emb.shape[0]
    
    # Similarity matrix: (N, N)
    logits = (img_emb @ txt_emb.t()) / tau
    labels = torch.arange(N, device=img_emb.device)
    
    # Image→Text and Text→Image losses
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.t(), labels)
    
    return (loss_i2t + loss_t2i) / 2


# Create dataset and dataloaders
dataset = SyntheticImageTextDataset(n_per_class=32, n_classes=4)
print(f"Dataset: {len(dataset)} image-text pairs")

batch_size = 64
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Create encoders
vision_encoder = VisionEncoder(input_dim=2, hidden_dim=16, embedding_dim=8).to(device)
text_encoder = TextEncoder(input_dim=2, hidden_dim=16, embedding_dim=8).to(device)

# Optimizer
optimizer = optim.Adam(list(vision_encoder.parameters()) + list(text_encoder.parameters()), lr=1e-3)

# Capture embeddings before training
vision_encoder.eval()
text_encoder.eval()
with torch.no_grad():
    img_emb_before = vision_encoder(dataset.images.to(device)).cpu().numpy()
    txt_emb_before = text_encoder(dataset.captions.to(device)).cpu().numpy()

# Training
losses = []
vision_encoder.train()
text_encoder.train()
for epoch in range(50):
    epoch_loss = 0.0
    for batch in dataloader:
        images = batch['image'].to(device)
        captions = batch['caption'].to(device)
        
        img_emb = vision_encoder(images)
        txt_emb = text_encoder(captions)
        
        loss = clip_loss_torch(img_emb, txt_emb, tau=0.07)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f}")

# Capture embeddings after training
vision_encoder.eval()
text_encoder.eval()
with torch.no_grad():
    img_emb_after = vision_encoder(dataset.images.to(device)).cpu().numpy()
    txt_emb_after = text_encoder(dataset.captions.to(device)).cpu().numpy()

# Visualize alignment: before vs after
pca = PCA(n_components=2)
labels = dataset.labels.numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['red', 'blue', 'green', 'orange']

# Before training
img_before_2d = pca.fit_transform(img_emb_before)
txt_before_2d = pca.fit_transform(txt_emb_before)
for cls in range(4):
    mask = labels == cls
    axes[0].scatter(img_before_2d[mask, 0], img_before_2d[mask, 1], 
                   c=colors[cls], s=20, alpha=0.6, label=f'Class {cls} (img)')
    axes[0].scatter(txt_before_2d[mask, 0], txt_before_2d[mask, 1], 
                   c=colors[cls], s=20, alpha=0.6, marker='x', label=f'Class {cls} (txt)')
axes[0].set_title("Before CLIP Training")
axes[0].set_xlabel("PCA dim 1")
axes[0].set_ylabel("PCA dim 2")
axes[0].legend(fontsize=8, loc='best')

# After training
img_after_2d = pca.fit_transform(img_emb_after)
txt_after_2d = pca.fit_transform(txt_emb_after)
for cls in range(4):
    mask = labels == cls
    axes[1].scatter(img_after_2d[mask, 0], img_after_2d[mask, 1], 
                   c=colors[cls], s=20, alpha=0.6, label=f'Class {cls} (img)')
    axes[1].scatter(txt_after_2d[mask, 0], txt_after_2d[mask, 1], 
                   c=colors[cls], s=20, alpha=0.6, marker='x', label=f'Class {cls} (txt)')
axes[1].set_title("After CLIP Training (50 epochs)")
axes[1].set_xlabel("PCA dim 1")
axes[1].set_ylabel("PCA dim 2")
axes[1].legend(fontsize=8, loc='best')

plt.tight_layout()
plt.savefig("/tmp/clip_alignment.png", dpi=80, bbox_inches='tight')
plt.show()
print(f"\nFinal loss: {losses[-1]:.4f} (started at {losses[0]:.4f})")
print("Alignment visualization saved to /tmp/clip_alignment.png")

## Real-World Example 1: Zero-Shot Classification with Pre-trained CLIP

Use transformers' pre-trained CLIP model for zero-shot image classification.
Encode class names and compute similarity to image embeddings.

In [ ]:
# Load pre-trained CLIP (small model for fast inference)
try:
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    print("Loaded pre-trained CLIP model")
except Exception as e:
    print(f"Error loading CLIP: {e}. Using fallback.")
    # Fallback: use our trained model
    model = None

if model is not None:
    model.to(device)
    model.eval()
    
    # Create synthetic images for demo
    # In practice, load real images
    def generate_synthetic_image_embedding():
        """Generate a random embedding as proxy for image."""
        return np.random.randn(768).astype(np.float32)  # ViT-B/32 has 512 or 768 D
    
    # Define class names with and without prompts
    classes = ["dog", "cat", "bird", "car"]
    
    # Prompt templates
    templates = [
        "a photo of a {}",
        "a {}",
        "an image of a {}",
    ]
    
    # Encode class names with multiple templates
    class_embeddings = []
    with torch.no_grad():
        for cls in classes:
            embeddings_cls = []
            for template in templates:
                text = template.format(cls)
                inputs = processor(text=text, return_tensors="pt").to(device)
                outputs = model.get_text_features(**inputs)
                outputs = outputs / outputs.norm(dim=-1, keepdim=True)
                embeddings_cls.append(outputs.cpu().numpy())
            # Average embeddings across templates
            avg_embedding = np.mean(embeddings_cls, axis=0)
            class_embeddings.append(avg_embedding)
    
    class_embeddings = np.vstack(class_embeddings)  # (4, D)
    print(f"Encoded {len(classes)} classes with {len(templates)} templates each")
    print(f"Class embedding shape: {class_embeddings.shape}")
    
    # Simulate zero-shot classification on synthetic batch
    print("\n=== Zero-Shot Classification (Simulated) ===")
    n_test = 4
    test_embeddings = []
    
    # In practice, encode real test images
    # For demo: use class embeddings as proxies (they're semantically aligned)
    for i in range(n_test):
        test_embeddings.append(class_embeddings[i % len(classes)] + 
                              np.random.randn(1, class_embeddings.shape[1]) * 0.01)
    test_embeddings = np.vstack(test_embeddings)
    
    # Compute similarities
    similarities = test_embeddings @ class_embeddings.T  # (n_test, n_classes)
    predictions = similarities.argmax(axis=1)
    confidences = similarities.max(axis=1)
    
    for i in range(n_test):
        pred_class = classes[predictions[i]]
        confidence = confidences[i]
        print(f"  Image {i+1}: predicted {pred_class} (confidence {confidence:.3f})")
else:
    print("Skipping pre-trained CLIP demo (model not loaded)")

## Real-World Example 2: Prompt Engineering Impact on Zero-Shot Accuracy

Demonstrate how prompt phrasing affects classification accuracy.
Compare bare class names vs descriptive prompts.

In [ ]:
# Simulate prompt engineering impact
def simulate_prompt_variation(base_embedding, num_prompts=5, noise_scale=0.05):
    """Simulate how different prompts produce slightly different embeddings.
    
    In reality, "a dog" and "a photo of a dog" have meaningfully different embeddings.
    Here we simulate by adding noise (high noise = more prompt variation).
    """
    embeddings = []
    for _ in range(num_prompts):
        noisy_emb = base_embedding + np.random.randn(*base_embedding.shape) * noise_scale
        embeddings.append(noisy_emb / np.linalg.norm(noisy_emb))
    return np.array(embeddings)

# Simulate a zero-shot classification task
n_classes = 5
n_test_images = 100
imaging_dim = 512

# Generate class embeddings
class_embs = []
for i in range(n_classes):
    emb = np.random.randn(imaging_dim)
    class_embs.append(emb / np.linalg.norm(emb))
class_embs = np.array(class_embs)

# Generate test images (aligned to one of the classes + noise)
true_labels = np.random.randint(0, n_classes, n_test_images)
test_embs = []
for label in true_labels:
    emb = class_embs[label] + np.random.randn(imaging_dim) * 0.1
    test_embs.append(emb / np.linalg.norm(emb))
test_embs = np.array(test_embs)

# Evaluate with different prompt engineering strategies
prompt_strategies = {
    'Bare class': 0.00,      # No variation (perfect prompts)
    'Single prompt': 0.02,   # Minimal noise
    'Moderate variation': 0.05,  # Medium noise
    'High variation': 0.10,  # High noise (poor prompt consistency)
}

results_prompt = {}
for strategy_name, noise_scale in prompt_strategies.items():
    # Get prompt-varied embeddings for each class
    class_embeddings_varied = []
    for cls_emb in class_embs:
        varied = simulate_prompt_variation(cls_emb.reshape(1, -1), num_prompts=3, noise_scale=noise_scale)
        avg_emb = varied.mean(axis=0)
        class_embeddings_varied.append(avg_emb)
    class_embeddings_varied = np.array(class_embeddings_varied)
    
    # Classify test images
    similarities = test_embs @ class_embeddings_varied.T
    predictions = similarities.argmax(axis=1)
    accuracy = (predictions == true_labels).mean()
    results_prompt[strategy_name] = accuracy
    print(f"{strategy_name:20s}: {accuracy:.4f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart: accuracy by strategy
strategies = list(results_prompt.keys())
accuracies = list(results_prompt.values())
colors_bars = ['green', 'blue', 'orange', 'red']
axes[0].bar(range(len(strategies)), accuracies, color=colors_bars, alpha=0.7)
axes[0].set_xticks(range(len(strategies)))
axes[0].set_xticklabels(strategies, rotation=15, ha='right')
axes[0].set_ylabel("Zero-Shot Accuracy")
axes[0].set_title("Prompt Engineering Impact on Accuracy")
axes[0].set_ylim([0, 1.0])
axes[0].grid(axis='y', alpha=0.3)

# Degradation from perfect prompts
baseline_acc = results_prompt['Bare class']
degradations = [baseline_acc - acc for acc in accuracies]
axes[1].bar(range(len(strategies)), degradations, color=colors_bars, alpha=0.7)
axes[1].set_xticks(range(len(strategies)))
axes[1].set_xticklabels(strategies, rotation=15, ha='right')
axes[1].set_ylabel("Accuracy Drop from Baseline")
axes[1].set_title("Cost of Poor Prompt Engineering")
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/clip_prompt_engineering.png", dpi=80, bbox_inches='tight')
plt.show()
print("\nPlot saved to /tmp/clip_prompt_engineering.png")

## Real-World Example 3: Temperature and Batch Size Effects on Contrastive Learning

Analyze how temperature (τ) and batch size affect CLIP training quality.
Temperature controls the sharpness of the similarity distribution;
batch size determines how many negatives the model sees per update.

In [ ]:
# Simulate effect of temperature and batch size
def alignment_metric(img_emb, txt_emb):
    """Alignment: average distance between matched pairs (lower = better)."""
    diffs = img_emb - txt_emb  # (N, D)
    distances = np.linalg.norm(diffs, axis=1)
    return distances.mean()


def uniformity_metric(embeddings):
    """Uniformity: how spread out embeddings are on unit sphere.
    
    Lower = more uniform (closer to -inf is ideal).
    """
    # Pairwise distances
    sq_dists = np.sum((embeddings[:, np.newaxis, :] - embeddings[np.newaxis, :, :]) ** 2, axis=2)
    # Gaussian kernel
    kernel_vals = np.exp(-2 * sq_dists)
    # Exclude diagonal
    mask = ~np.eye(len(embeddings), dtype=bool)
    return np.log(kernel_vals[mask].mean())


# Sweep temperature values
temps = [0.01, 0.05, 0.07, 0.1, 0.5]
batch_sizes = [32, 64, 256, 1024]

results_temp_batch = {}
for temp in temps:
    for batch_size in batch_sizes:
        # Simulate training: more batches → better alignment
        # Temperature controls how sharp the learning is
        
        # Generate training embeddings
        n_pairs = batch_size
        D = 128
        base = np.random.randn(n_pairs, D).astype(np.float32)
        
        # More training steps (larger batch = faster convergence in simulation)
        num_steps = max(5, 20 // (batch_size // 32))  # More steps for small batches
        
        img_emb = base + np.random.randn(n_pairs, D) * 0.3
        txt_emb = base.copy()
        
        # Simulate convergence: alignment improves with more steps
        for step in range(num_steps):
            # Contrastive gradient: pull img_emb closer to txt_emb
            alignment = alignment_metric(img_emb, txt_emb)
            # Temperature: controls how aggressive the update is
            # Lower tau = sharper distribution = more aggressive update
            step_size = temp * 0.1  # Larger temp → larger steps (paradoxically)
            img_emb += (txt_emb - img_emb) * step_size
        
        align_final = alignment_metric(img_emb, txt_emb)
        unif_final = uniformity_metric(img_emb)
        
        results_temp_batch[(temp, batch_size)] = {
            'alignment': align_final,
            'uniformity': unif_final,
        }

# Print results
print("\n=== Temperature and Batch Size Effects ===")
print(f"{'Temp':>6} | {'Batch Size':>10} | {'Alignment':>10} | {'Uniformity':>10}")
print("-" * 50)
for temp in temps:
    for batch_size in batch_sizes:
        key = (temp, batch_size)
        alignment = results_temp_batch[key]['alignment']
        uniformity = results_temp_batch[key]['uniformity']
        print(f"{temp:>6.3f} | {batch_size:>10} | {alignment:>10.4f} | {uniformity:>10.4f}")

# Visualize: alignment vs batch size for different temperatures
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for temp in temps:
    alignments = [results_temp_batch[(temp, bs)]['alignment'] for bs in batch_sizes]
    axes[0].plot(batch_sizes, alignments, marker='o', label=f'tau={temp}', linewidth=2)

axes[0].set_xscale('log')
axes[0].set_xlabel("Batch Size (log scale)")
axes[0].set_ylabel("Alignment (lower = better)")
axes[0].set_title("Temperature Impact on Alignment vs Batch Size")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Heatmap: alignment by temp × batch_size
alignment_matrix = np.zeros((len(temps), len(batch_sizes)))
for i, temp in enumerate(temps):
    for j, batch_size in enumerate(batch_sizes):
        alignment_matrix[i, j] = results_temp_batch[(temp, batch_size)]['alignment']

im = axes[1].imshow(alignment_matrix, cmap='viridis', aspect='auto')
axes[1].set_xticks(range(len(batch_sizes)))
axes[1].set_yticks(range(len(temps)))
axes[1].set_xticklabels([str(bs) for bs in batch_sizes])
axes[1].set_yticklabels([f'{t:.2f}' for t in temps])
axes[1].set_xlabel("Batch Size")
axes[1].set_ylabel("Temperature")
axes[1].set_title("Alignment Heatmap")
plt.colorbar(im, ax=axes[1], label="Alignment Score")

plt.tight_layout()
plt.savefig("/tmp/clip_temp_batch.png", dpi=80, bbox_inches='tight')
plt.show()
print("\nPlot saved to /tmp/clip_temp_batch.png")

## Comparison: Zero-Shot CLIP vs Fine-Tuned Linear Classifier

Compare zero-shot transfer (using CLIP directly) vs fine-tuning a linear classifier on top of frozen embeddings.
Show when zero-shot wins and when fine-tuning is necessary.

In [ ]:
# Simulate zero-shot vs fine-tuned classifier
n_classes = 10
n_features = 512
n_train_per_class = 10  # Small dataset
n_test = 1000

# Generate synthetic CLIP embeddings
# Class embeddings from pre-training (aligned with text prompts)
class_embeddings_gt = np.random.randn(n_classes, n_features)
class_embeddings_gt = class_embeddings_gt / np.linalg.norm(class_embeddings_gt, axis=1, keepdims=True)

# Generate training data: samples from each class + domain shift
np.random.seed(42)
domain_shift_scale = 0.2
train_labels = np.repeat(np.arange(n_classes), n_train_per_class)
train_embs = []
for label in train_labels:
    # Sample from class embedding + small noise + domain shift
    emb = class_embeddings_gt[label] + np.random.randn(n_features) * 0.05 + \
          np.random.randn(n_features) * domain_shift_scale
    train_embs.append(emb / np.linalg.norm(emb))
train_embs = np.array(train_embs)

# Generate test data (same distribution as train, but with additional noise)
test_labels = np.random.randint(0, n_classes, n_test)
test_embs = []
for label in test_labels:
    emb = class_embeddings_gt[label] + np.random.randn(n_features) * 0.08 + \
          np.random.randn(n_features) * domain_shift_scale
    test_embs.append(emb / np.linalg.norm(emb))
test_embs = np.array(test_embs)

# Zero-shot classification: use ground-truth class embeddings
zeroshot_sims = test_embs @ class_embeddings_gt.T
zeroshot_preds = zeroshot_sims.argmax(axis=1)
zeroshot_acc = (zeroshot_preds == test_labels).mean()

# Fine-tuned linear classifier: train logistic regression on embeddings
clf = LogisticRegression(max_iter=500, random_state=42)
clf.fit(train_embs, train_labels)
finetune_preds = clf.predict(test_embs)
finetune_acc = (finetune_preds == test_labels).mean()

print("\n=== Zero-Shot vs Fine-Tuned Classifier ===")
print(f"Domain shift scale: {domain_shift_scale}")
print(f"Training samples: {len(train_labels)} ({n_train_per_class} per class)")
print(f"Test samples: {n_test}")
print(f"\nZero-shot accuracy: {zeroshot_acc:.4f}")
print(f"Fine-tuned (linear probe) accuracy: {finetune_acc:.4f}")
print(f"Fine-tuning gain: {(finetune_acc - zeroshot_acc):.4f} ({(finetune_acc - zeroshot_acc) / zeroshot_acc * 100:.1f}%)")

# Sweep: vary training set size to show when fine-tuning becomes beneficial
train_sizes = [5, 10, 20, 50, 100, 200]
zeroshot_accs = []
finetune_accs = []

for train_per_cls in train_sizes:
    train_labels_var = np.repeat(np.arange(n_classes), train_per_cls)
    train_embs_var = []
    for label in train_labels_var:
        emb = class_embeddings_gt[label] + np.random.randn(n_features) * 0.05 + \
              np.random.randn(n_features) * domain_shift_scale
        train_embs_var.append(emb / np.linalg.norm(emb))
    train_embs_var = np.array(train_embs_var)
    
    # Zero-shot (constant, doesn't depend on train size)
    zeroshot_accs.append(zeroshot_acc)
    
    # Fine-tune
    clf_var = LogisticRegression(max_iter=500, random_state=42)
    clf_var.fit(train_embs_var, train_labels_var)
    ft_acc = clf_var.score(test_embs, test_labels)
    finetune_accs.append(ft_acc)

# Visualize: accuracy vs training set size
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_sizes, zeroshot_accs, marker='o', linewidth=2, markersize=8, 
        label="Zero-shot CLIP", color='steelblue')
ax.plot(train_sizes, finetune_accs, marker='s', linewidth=2, markersize=8, 
        label="Fine-tuned (linear probe)", color='coral')
ax.fill_between(train_sizes, zeroshot_accs, finetune_accs, alpha=0.2, color='gray')
ax.set_xlabel("Training Samples per Class")
ax.set_ylabel("Accuracy on Test Set")
ax.set_title("Zero-Shot vs Fine-Tuned: Sample Efficiency")
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim([0, 1.0])

plt.tight_layout()
plt.savefig("/tmp/clip_zeroshot_vs_finetuned.png", dpi=80, bbox_inches='tight')
plt.show()
print("\nPlot saved to /tmp/clip_zeroshot_vs_finetuned.png")

## Key Takeaways

**Core idea:**
CLIP learns a shared embedding space for images and text through contrastive learning on web-scale paired data.
This enables zero-shot transfer: classify new images by encoding class names and finding nearest matches.

**Key mechanisms:**
1. **Vision + Text encoders:** Project to shared embedding space, L2-normalized
2. **Contrastive loss (InfoNCE):** Maximize sim(image_i, text_i), minimize sim(image_i, text_j) for i≠j
3. **Large-scale data:** 400M image-text pairs teach general visual concepts
4. **Zero-shot transfer:** Encode class names, find nearest image embeddings (no fine-tuning)

**Critical trade-offs:**
- **Temperature τ:** Lower (0.01) = sharp distribution, fast learning, but unstable. Higher (0.5) = smooth, stable, slow. Default 0.07 balances both.
- **Batch size:** Larger batches (32K) provide more negative pairs, better convergence. Small batches (32) are simple but slow. For fine-tuning, 256+ is typical.
- **Prompt engineering:** "a photo of a {class}" >> "{class}". Prompt ensembling improves accuracy 2–5%. Spend time on domain-specific prompts.
- **Zero-shot vs fine-tuning:** Zero-shot excellent for diverse domains (low training data, high domain shift). Fine-tuning wins when you have labeled data in-domain.

**Common pitfalls:**
1. Poor prompts: Using bare class names instead of natural language descriptions
2. Domain mismatch: Pre-trained on web images; specialized domains (medical, satellite) need fine-tuning
3. Over-fitting during fine-tuning: Degrades zero-shot transfer. Use small learning rates and early stopping.
4. Missing normalization: Embeddings must be L2-normalized before cosine similarity
5. Hallucinating on out-of-distribution images: Add confidence thresholds to abstain on uncertain cases

**Best practices:**
- Start zero-shot before fine-tuning
- Use prompt ensembling for improved accuracy
- Freeze text encoder, fine-tune vision encoder for domain adaptation
- Cache embeddings for fast inference
- Monitor alignment and uniformity metrics during training